In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🚦 Rate Limiting & Traffic Shaping Guide

**Protecting systems with advanced rate limiting, traffic shaping, and congestion control**

This guide covers:
- Rate limiting algorithms
- Token bucket, leaky bucket patterns
- Sliding window implementations
- Distributed rate limiting
- Traffic shaping strategies
- DDoS mitigation
- Queue management
- Adaptive throttling
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Rate Limiting Algorithms

### Token Bucket Algorithm
```python
from dataclasses import dataclass
from typing import Dict, List, Optional
from datetime import datetime, timedelta
import math

@dataclass
class RateLimitConfig:
    """Configuration for rate limiting"""
    capacity: int  # Max tokens
    refill_rate: float  # Tokens per second
    window_size_seconds: int = 1

class TokenBucket:
    """Token bucket rate limiter"""
    
    def __init__(self, capacity: int, refill_rate: float):
        self.capacity = capacity
        self.refill_rate = refill_rate  # tokens per second
        self.tokens = float(capacity)
        self.last_refill = datetime.now()
    
    def refill(self):
        """Refill tokens based on time elapsed"""
        
        now = datetime.now()
        elapsed = (now - self.last_refill).total_seconds()
        
        tokens_to_add = elapsed * self.refill_rate
        self.tokens = min(self.capacity, self.tokens + tokens_to_add)
        
        self.last_refill = now
    
    def try_consume(self, tokens_needed: int = 1) -> bool:
        """Try to consume tokens"""
        
        self.refill()
        
        if self.tokens >= tokens_needed:
            self.tokens -= tokens_needed
            return True
        
        return False
    
    def get_remaining_tokens(self) -> float:
        """Get remaining tokens"""
        self.refill()
        return self.tokens
    
    def wait_time_for_tokens(self, tokens_needed: int) -> float:
        """Calculate wait time for required tokens"""
        
        self.refill()
        
        if self.tokens >= tokens_needed:
            return 0.0
        
        tokens_needed_to_wait = tokens_needed - self.tokens
        wait_seconds = tokens_needed_to_wait / self.refill_rate
        
        return wait_seconds

class LeakyBucket:
    """Leaky bucket rate limiter"""
    
    def __init__(self, capacity: int, leak_rate: float):
        self.capacity = capacity
        self.leak_rate = leak_rate  # items per second
        self.queue: List[Dict] = []
        self.last_leak = datetime.now()
    
    def leak(self):
        """Remove items from queue (leak)"""
        
        now = datetime.now()
        elapsed = (now - self.last_leak).total_seconds()
        
        items_to_leak = int(elapsed * self.leak_rate)
        
        for _ in range(items_to_leak):
            if self.queue:
                self.queue.pop(0)
        
        self.last_leak = now
    
    def add_item(self, item: Dict) -> bool:
        """Try to add item to queue"""
        
        self.leak()
        
        if len(self.queue) < self.capacity:
            self.queue.append(item)
            return True
        
        return False
    
    def get_queue_status(self) -> Dict:
        """Get queue status"""
        
        self.leak()
        
        return {
            'queue_size': len(self.queue),
            'capacity': self.capacity,
            'utilization_percent': (len(self.queue) / self.capacity) * 100 if self.capacity > 0 else 0
        }

class SlidingWindowCounter:
    """Sliding window rate limiter"""
    
    def __init__(self, capacity: int, window_seconds: int):
        self.capacity = capacity
        self.window_seconds = window_seconds
        self.requests: List[datetime] = []
    
    def cleanup_old_requests(self):
        """Remove requests outside window"""
        
        now = datetime.now()
        cutoff = now - timedelta(seconds=self.window_seconds)
        
        self.requests = [req for req in self.requests if req > cutoff]
    
    def try_consume(self) -> bool:
        """Try to consume one request"""
        
        self.cleanup_old_requests()
        
        if len(self.requests) < self.capacity:
            self.requests.append(datetime.now())
            return True
        
        return False
    
    def get_request_count(self) -> int:
        """Get current request count"""
        self.cleanup_old_requests()
        return len(self.requests)
    
    def get_available_capacity(self) -> int:
        """Get available capacity"""
        self.cleanup_old_requests()
        return self.capacity - len(self.requests)

class SlidingWindowLog:
    """Sliding window log rate limiter (more accurate)"""
    
    def __init__(self, capacity: int, window_seconds: int):
        self.capacity = capacity
        self.window_seconds = window_seconds
        self.logs: List[datetime] = []
    
    def is_allowed(self) -> bool:
        """Check if request is allowed"""
        
        now = datetime.now()
        cutoff = now - timedelta(seconds=self.window_seconds)
        
        # Remove old entries
        self.logs = [log for log in self.logs if log > cutoff]
        
        if len(self.logs) < self.capacity:
            self.logs.append(now)
            return True
        
        return False
    
    def get_rate_status(self) -> Dict:
        """Get rate limit status"""
        
        now = datetime.now()
        cutoff = now - timedelta(seconds=self.window_seconds)
        
        self.logs = [log for log in self.logs if log > cutoff]
        
        return {
            'requests_in_window': len(self.logs),
            'capacity': self.capacity,
            'utilization_percent': (len(self.logs) / self.capacity) * 100 if self.capacity > 0 else 0
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Distributed Rate Limiting

### Distributed Token Bucket
```python
class DistributedRateLimiter:
    """Rate limiter for distributed systems"""
    
    def __init__(self, identifier: str, capacity: int, refill_rate: float):
        self.identifier = identifier
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.local_bucket = TokenBucket(capacity, refill_rate)
        self.remote_state: Optional[Dict] = None
    
    def sync_with_remote(self, remote_state: Dict) -> bool:
        """Sync state with remote rate limiter"""
        
        self.remote_state = remote_state
        
        # Merge local and remote token counts
        local_tokens = self.local_bucket.get_remaining_tokens()
        remote_tokens = remote_state.get('tokens', 0)
        
        # Use the minimum to ensure consistency
        merged_tokens = min(local_tokens, remote_tokens)
        self.local_bucket.tokens = merged_tokens
        
        return True
    
    def try_consume(self, tokens: int = 1) -> bool:
        """Try to consume tokens (with remote sync)"""
        
        # First try local
        if self.local_bucket.try_consume(tokens):
            return True
        
        # Could attempt remote sync here
        return False

class RateLimiterCluster:
    """Coordinate rate limiting across cluster"""
    
    def __init__(self):
        self.limiters: Dict[str, DistributedRateLimiter] = {}
        self.central_state: Dict[str, Dict] = {}
    
    def register_limiter(self, limiter_id: str, limiter: DistributedRateLimiter):
        """Register rate limiter"""
        self.limiters[limiter_id] = limiter
    
    def sync_all(self):
        """Synchronize all limiters"""
        
        for limiter_id, limiter in self.limiters.items():
            # Get current state
            state = {
                'tokens': limiter.local_bucket.get_remaining_tokens(),
                'capacity': limiter.capacity
            }
            
            # Merge with central state
            if limiter_id in self.central_state:
                central_tokens = self.central_state[limiter_id].get('tokens', 0)
                state['tokens'] = min(state['tokens'], central_tokens)
            
            self.central_state[limiter_id] = state
            limiter.sync_with_remote(state)
    
    def get_cluster_status(self) -> Dict:
        """Get cluster rate limiter status"""
        
        return {
            'limiters': len(self.limiters),
            'state': self.central_state
        }
```

### Multi-Tenant Rate Limiting
```python
class TenantRateLimiter:
    """Rate limiting per tenant"""
    
    def __init__(self):
        self.tenant_limiters: Dict[str, TokenBucket] = {}
        self.global_capacity = 100000  # Global limit
        self.global_bucket = TokenBucket(self.global_capacity, 1000)
    
    def get_or_create_limiter(self, tenant_id: str, tenant_capacity: int) -> TokenBucket:
        """Get or create tenant limiter"""
        
        if tenant_id not in self.tenant_limiters:
            self.tenant_limiters[tenant_id] = TokenBucket(tenant_capacity, tenant_capacity / 60)
        
        return self.tenant_limiters[tenant_id]
    
    def is_allowed(self, tenant_id: str) -> bool:
        """Check if request allowed for tenant"""
        
        # Check global limit
        if not self.global_bucket.try_consume(1):
            return False
        
        # Check tenant limit
        limiter = self.get_or_create_limiter(tenant_id, 1000)
        
        return limiter.try_consume(1)
    
    def get_tenant_status(self, tenant_id: str) -> Dict:
        """Get tenant rate limit status"""
        
        limiter = self.tenant_limiters.get(tenant_id)
        
        if not limiter:
            return {}
        
        return {
            'tenant_id': tenant_id,
            'available_tokens': limiter.get_remaining_tokens(),
            'capacity': limiter.capacity
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Traffic Shaping & Congestion Control

### Adaptive Rate Limiting
```python
class AdaptiveRateLimiter:
    """Adapt rate limits based on system load"""
    
    def __init__(self, initial_capacity: int):
        self.base_capacity = initial_capacity
        self.current_capacity = initial_capacity
        self.limiter = TokenBucket(initial_capacity, initial_capacity / 60)
        self.load_history: List[float] = []
    
    def adjust_for_load(self, current_load_percent: float):
        """Adjust capacity based on system load"""
        
        self.load_history.append(current_load_percent)
        
        # Keep recent history
        if len(self.load_history) > 60:
            self.load_history.pop(0)
        
        # Calculate average load
        avg_load = sum(self.load_history) / len(self.load_history)
        
        if avg_load > 80:
            # Reduce capacity
            self.current_capacity = int(self.base_capacity * 0.7)
        elif avg_load > 60:
            # Slightly reduce
            self.current_capacity = int(self.base_capacity * 0.85)
        elif avg_load < 30:
            # Increase capacity
            self.current_capacity = self.base_capacity
        
        # Recreate limiter with new capacity
        self.limiter = TokenBucket(self.current_capacity, self.current_capacity / 60)
    
    def is_allowed(self) -> bool:
        """Check if request allowed"""
        return self.limiter.try_consume(1)

class QueueManager:
    """Manage request queue under load"""
    
    def __init__(self, max_queue_size: int = 1000):
        self.max_queue_size = max_queue_size
        self.queue: List[Dict] = []
        self.dropped_count = 0
    
    def enqueue(self, request: Dict) -> bool:
        """Try to enqueue request"""
        
        if len(self.queue) < self.max_queue_size:
            self.queue.append(request)
            return True
        
        self.dropped_count += 1
        return False
    
    def dequeue(self, batch_size: int = 10) -> List[Dict]:
        """Dequeue requests for processing"""
        
        batch = self.queue[:batch_size]
        self.queue = self.queue[batch_size:]
        
        return batch
    
    def get_queue_status(self) -> Dict:
        """Get queue status"""
        
        return {
            'queue_size': len(self.queue),
            'max_size': self.max_queue_size,
            'utilization_percent': (len(self.queue) / self.max_queue_size) * 100,
            'dropped_requests': self.dropped_count
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Rate Limiting & Traffic Shaping Checklist

✅ **Algorithm Selection**
- [ ] Algorithm chosen (token bucket, leaky bucket, sliding window)
- [ ] Capacity configured
- [ ] Refill/leak rate set
- [ ] Window size appropriate
- [ ] Performance tested

✅ **Implementation**
- [ ] Rate limiter deployed
- [ ] Distributed consistency
- [ ] Multi-tenant support
- [ ] Client integration
- [ ] Error handling

✅ **Monitoring**
- [ ] Metrics tracked
- [ ] Queue depth monitored
- [ ] Drop rate observed
- [ ] Load patterns analyzed
- [ ] Alerts configured

✅ **DDoS Protection**
- [ ] Rate limits aggressive
- [ ] IP-based limiting
- [ ] Fingerprinting enabled
- [ ] Fallback strategy
- [ ] Escalation procedure

✅ **Operational**
- [ ] Configuration manageable
- [ ] Tuneable parameters
- [ ] Graceful degradation
- [ ] Recovery strategy
- [ ] Documentation complete
</VSCode.Cell>
```